# NUS prerequisite graph construction

This notebook builds prerequisite-based outputs used in downstream curriculum analysis.

## Purpose
- Fetch the NUS module catalogue for AY `2025-2026`
- Filter to graded undergraduate modules with non-empty description, faculty, and department
- Query NUSMods for module prerequisite trees
- Build a reverse prerequisite map from prerequisite token to dependent modules
- Handle wildcard prerequisite tokens
- Deduplicate dependent modules by title/description identity

## Key outputs
- `nus_prerequisite_graph.json` — mapping from **module** to its direct prerequisites
- `dependent_mods_dedup.csv` — reverse prerequisite table with deduplicated dependent modules

## Notes
- This notebook preserves the original project logic and output formats.
- Downstream notebooks may reverse `nus_prerequisite_graph.json` again to obtain prerequisite-to-dependents mappings.


## 1. Imports

In [ ]:
import json
import os
import pickle
import re
import time
from collections import defaultdict
from typing import Any, Dict, List, Optional, Set

import pandas as pd
import requests

## 2. Load and filter the NUS module catalogue

In [ ]:
MODULE_LIST_URL = "https://api.nusmods.com/v2/2025-2026/moduleInfo.json"

response = requests.get(MODULE_LIST_URL, timeout=30)
response.raise_for_status()
data = response.json()

In [ ]:
def is_undergrad(code: str) -> bool:
    match = re.search(r"\d+", str(code))
    # Returns True if it has numbers and doesn't start with 5
    return bool(match) and not match.group().startswith("5")

filtered_data = []

for d in data:
    # 1. Check grading basis and undergrad status
    if d.get("gradingBasisDescription") == "Graded" and is_undergrad(d.get("moduleCode")):
        # 2. Extract values safely
        desc = str(d.get("description", "")).strip()
        fac = str(d.get("faculty", "")).strip()
        dept = str(d.get("department", "")).strip()

        # 3. Ensure all three are not empty/whitespace
        if desc and fac and dept:
            filtered_data.append({
                "moduleCode": d.get("moduleCode"),
                "title": d.get("title"),
                "description": desc,
                "faculty": fac,
                "department": dept
            })

module_base_df = pd.DataFrame(filtered_data)
module_base_df.rename(columns={"moduleCode": "module_code"}, inplace=True)
module_base_df["module_code"] = module_base_df["module_code"].astype(str).str.strip()
module_base_df = module_base_df.drop_duplicates(subset=["module_code"]).reset_index(drop=True)

valid_module_codes: Set[str] = set(module_base_df["module_code"])

## 3. Configure API session and cache

In [ ]:
session = requests.Session()
session.headers.update({"Accept": "application/json"})

def save_cache(cache: Dict[str, Any], path: str = "data/prereq_cache.pkl") -> None:
    os.makedirs("data", exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(cache, f)

def load_cache(path: str = "data/prereq_cache.pkl") -> Dict[str, Any]:
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return {}

prereq_cache = load_cache()
print(f"Loaded cache with {len(prereq_cache)} modules")

## 4. Fetch prerequisite trees

In [ ]:
def fetch_prereq_info(module_code: str, acad_year: str = "2025-2026") -> Optional[Dict[str, Any]]:
    """
    Fetch prereqTree for a module, trying the given academic year first,
    then earlier years down to 2020-2021.

    Returns:
        prereqTree if found, else None
    """
    latest_year = int(acad_year.split("-")[0])

    while latest_year >= 2020:
        curr_acad_year = f"{latest_year}-{latest_year + 1}"
        url = f"https://api.nusmods.com/v2/{curr_acad_year}/modules/{module_code}.json"

        try:
            response = session.get(url, timeout=60)
            if response.status_code == 404:
                latest_year -= 1
                continue

            response.raise_for_status()
            module_json = response.json()
            return module_json.get("prereqTree")

        except requests.exceptions.RequestException:
            latest_year -= 1
            continue

    return None

def normalize_module_code(code: str) -> str:
    """Remove suffixes like :D from module codes."""
    if ":" in code:
        return code.split(":")[0]
    return code

def extract_all_prereqs(prereq_tree: Dict) -> Set[str]:
    """Recursively extract all module codes from prerequisite tree."""
    if not prereq_tree:
        return set()

    prereqs = set()

    if isinstance(prereq_tree, str):
        prereqs.add(normalize_module_code(prereq_tree))

    elif isinstance(prereq_tree, dict):
        for value in prereq_tree.values():
            prereqs.update(extract_all_prereqs(value))

    elif isinstance(prereq_tree, list):
        for item in prereq_tree:
            prereqs.update(extract_all_prereqs(item))

    return prereqs

def build_direct_prereqs(module_codes: List[str], cache: Dict[str, Any], save_every: int = 500) -> Dict[str, List[str]]:
    result = {}

    for i, module_code in enumerate(module_codes, start=1):
        print(f"[{i}/{len(module_codes)}] {module_code}")

        if module_code in cache:
            prereq_tree = cache[module_code]
        else:
            try:
                prereq_tree = fetch_prereq_info(module_code)
            except Exception as e:
                print(f"Failed for {module_code}: {e}")
                prereq_tree = None

            cache[module_code] = prereq_tree

        if not prereq_tree:
            result[module_code] = []
        else:
            result[module_code] = sorted(extract_all_prereqs(prereq_tree))

        if i % save_every == 0:
            save_cache(cache)
            print(f"Checkpoint saved at {i} modules")
            time.sleep(0.1)

    save_cache(cache)
    return result

In [ ]:
module_codes = module_base_df["module_code"].tolist()
len(module_codes)

In [ ]:
direct_prereqs = build_direct_prereqs(
    module_codes,
    cache=prereq_cache
)

## 5. Save and reload direct prerequisites

In [ ]:
with open("nus_prerequisite_graph.json", "w") as file:
    json.dump(direct_prereqs, file, indent=2)

print("Saved direct_prereqs JSON to nus_prerequisite_graph.json")

In [ ]:
# Read back in after initial run if needed
with open("nus_prerequisite_graph.json", "r") as file:
    direct_prereqs = json.load(file)

## 6. Build reverse dependency map

In [ ]:
reverse_map = defaultdict(set)

for module_code, prereqs in direct_prereqs.items():
    if not prereqs:
        continue

    for prereq in set(prereqs):   
        reverse_map[prereq].add(module_code)

Wildcard modules are those with `%` behind to specify different courses.  
It was investigated that some variants point to the same module while others align to different modules for different degrees.

In [ ]:
reverse_count_df = pd.DataFrame([
    {
        "prereq_token": prereq,
        "dependent_modules": sorted(dependents)
    }
    for prereq, dependents in reverse_map.items()
])

reverse_count_df["is_wildcard"] = reverse_count_df["prereq_token"].str.contains("%", regex=False)
reverse_count_df.head(20)

In [ ]:
def classify_wildcard(row):
    token = row["prereq_token"]

    # Only classify wildcard rows
    if not row["is_wildcard"]:
        return None

    prefix = token.replace("%", "")

    # Extract digits
    digits = re.findall(r"\d", prefix)
    n_digits = len(digits)

    if n_digits == 4:
        return "narrow"
    else:
        return "broad"

reverse_count_df["wildcard_type"] = reverse_count_df.apply(classify_wildcard, axis=1)
reverse_count_df.head()

In [ ]:
focused_df = reverse_count_df[
    (reverse_count_df["is_wildcard"] == False) |
    (reverse_count_df["wildcard_type"] == "narrow")
].copy()

focused_df.head(20)

## 7. Prepare dependent module information

There may be double counting of modules in `dependent_modules`.

In [ ]:
all_dependent_modules = sorted({
    mod
    for mods in focused_df["dependent_modules"]
    if isinstance(mods, list)
    for mod in mods
})

In [ ]:
dependent_modules_df = module_base_df[module_base_df["module_code"].isin(all_dependent_modules)].copy()

The original workflow then reloads `dependent_mods_info.csv` before identity-based deduplication.  
This is preserved here to avoid changing the project logic.

In [ ]:
dependent_modules_df = pd.read_csv("dependent_mods_info.csv")

In [ ]:
def normalize_text_for_identity(text):
    if text is None:
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

dependent_modules_df["title_norm"] = dependent_modules_df["title"].apply(normalize_text_for_identity)
dependent_modules_df["description_norm"] = dependent_modules_df["description"].apply(normalize_text_for_identity)

dependent_modules_df["identity"] = (
    dependent_modules_df["title_norm"] + " || " + dependent_modules_df["description_norm"]
)

# Fallback if both missing
dependent_modules_df.loc[
    dependent_modules_df["identity"] == " || ", "identity"
] = "CODE_ONLY::" + dependent_modules_df["module_code"]

In [ ]:
module_to_identity = dict(
    zip(dependent_modules_df["module_code"], dependent_modules_df["identity"])
)

identity_to_rep = (
    dependent_modules_df.groupby("identity")["module_code"]
    .min()
    .to_dict()
)

In [ ]:
def deduplicate_dependent_modules(mod_list, module_to_identity, identity_to_rep):
    if not isinstance(mod_list, list):
        return []

    seen_identities = set()
    deduped = []

    for mod in mod_list:
        identity = module_to_identity.get(mod, f"CODE_ONLY::{mod}")
        if identity not in seen_identities:
            seen_identities.add(identity)
            deduped.append(identity_to_rep.get(identity, mod))

    return deduped

In [ ]:
focused_df["dependent_modules_dedup"] = focused_df["dependent_modules"].apply(
    lambda mods: deduplicate_dependent_modules(mods, module_to_identity, identity_to_rep)
)

## 8. Compute counts and export

In [ ]:
focused_df["n_direct_dependents_raw"] = focused_df["dependent_modules"].apply(
    lambda mods: len(mods) if isinstance(mods, list) else 0
)

focused_df["n_direct_dependents_dedup"] = focused_df["dependent_modules_dedup"].apply(len)

focused_df["inflation_removed"] = (
    focused_df["n_direct_dependents_raw"] - focused_df["n_direct_dependents_dedup"]
)

focused_df.sort_values(by="n_direct_dependents_dedup", ascending=False, inplace=True)
focused_df.head()

In [ ]:
focused_df.to_csv("dependent_mods_dedup.csv", index=False)
print("Saved dependent_mods_dedup.csv")